# FPembed Quick Start

This notebook demonstrates the basic workflow for generating compressed molecular fingerprint embeddings using the `fpembed` package.

`FPembed` supports six binary fingerprint types (ECFP, AtomPair, TopologicalTorsion, RDKit, Layered, Pattern) through a single unified class backed by `scikit-fingerprints`.

## 1. Setup

Install the package if you haven't already:
```bash
pip install fpembed
```

In [ ]:
from fpembed import EmbeddedFingerprintGenerator, compress_fingerprint, parse_smiles, fp_params_hash, __version__
import numpy as np

print(f"FPembed version: {__version__}")

## 2. Create an EmbeddedFingerprintGenerator

The generator holds your configuration (`fp_type`, `fp_size`, `compression`, and type-specific `fp_params`) and precomputes the weight mask once.

In [ ]:
gen = EmbeddedFingerprintGenerator(
    fp_type="ecfp", fp_size=2048, compression=16, fp_params={"radius": 2}
)
print(gen)

## 3. Single Molecule from SMILES

In [ ]:
emb = gen.GetFingerprintFromSmiles("CCO")  # ethanol
print(f"Shape: {emb.shape}")  # (128,) = 2048 / 16
print(f"First 10 values: {emb[:10]}")

## 4. Different Fingerprint Types

Switch fingerprint types by changing `fp_type` and providing the appropriate `fp_params`.

In [ ]:
# Atom Pair fingerprint
gen_ap = EmbeddedFingerprintGenerator(
    fp_type="atom_pair", fp_size=2048, compression=16,
    fp_params={"min_distance": 1, "max_distance": 30}
)
emb_ap = gen_ap.GetFingerprintFromSmiles("CCO")
print(f"AtomPair shape: {emb_ap.shape}")

# Topological Torsion fingerprint
gen_tt = EmbeddedFingerprintGenerator(
    fp_type="topological_torsion", fp_size=2048, compression=16,
    fp_params={"torsion_atom_count": 4}
)
emb_tt = gen_tt.GetFingerprintFromSmiles("CCO")
print(f"TopologicalTorsion shape: {emb_tt.shape}")

# RDKit fingerprint
gen_rdk = EmbeddedFingerprintGenerator(
    fp_type="rdkit", fp_size=2048, compression=16,
    fp_params={"min_path": 1, "max_path": 7}
)
emb_rdk = gen_rdk.GetFingerprintFromSmiles("CCO")
print(f"RDKit shape: {emb_rdk.shape}")

# Layered fingerprint
gen_lay = EmbeddedFingerprintGenerator(
    fp_type="layered", fp_size=2048, compression=16,
    fp_params={"min_path": 1, "max_path": 7}
)
emb_lay = gen_lay.GetFingerprintFromSmiles("CCO")
print(f"Layered shape: {emb_lay.shape}")

# Pattern fingerprint (no type-specific params)
gen_pat = EmbeddedFingerprintGenerator(
    fp_type="pattern", fp_size=2048, compression=16
)
emb_pat = gen_pat.GetFingerprintFromSmiles("CCO")
print(f"Pattern shape: {emb_pat.shape}")

## 5. Single Molecule from SELFIES

SELFIES strings are decoded to SMILES internally — same result as the equivalent SMILES input.

In [ ]:
emb_selfies = gen.GetFingerprintFromSelfies("[C][C][O]")
print(f"Shape: {emb_selfies.shape}")
print(f"Matches SMILES result: {np.array_equal(emb, emb_selfies)}")

## 6. Batch Processing

Process multiple molecules at once. Invalid entries are skipped and their indices returned.

In [ ]:
smiles_list = [
    "CCO",           # ethanol
    "c1ccccc1",      # benzene
    "CC(=O)O",       # acetic acid
    "invalid",       # invalid SMILES
    "CC(=O)Oc1ccccc1C(=O)O",  # aspirin
]

embeddings, invalid_indices = gen.GetFingerprintsFromSmiles(smiles_list)
print(f"Embeddings shape: {embeddings.shape}")  # (4, 128)
print(f"Invalid indices: {invalid_indices}")      # [3]

## 7. Raw Fingerprint (No Compression)

Set `compression=None` to get the uncompressed fingerprint.

In [ ]:
gen_raw = EmbeddedFingerprintGenerator(
    fp_type="ecfp", fp_size=2048, compression=None, fp_params={"radius": 2}
)
fp = gen_raw.GetFingerprintFromSmiles("CCO")
print(f"Raw FP shape: {fp.shape}")  # (2048,)

## 8. Using RDKit Mol Objects Directly

In [ ]:
mol = parse_smiles("c1ccccc1")
emb = gen.GetFingerprintAsNumPy(mol)
print(f"Shape: {emb.shape}")

## 9. Standalone Compression Function

`compress_fingerprint` applies log-space weighted binary masking to any fingerprint vector.

In [ ]:
fp = np.random.randint(0, 2, size=2048).astype(np.float64)
embedded = compress_fingerprint(fp, size=16)
print(f"Input shape: {fp.shape} -> Output shape: {embedded.shape}")

## 10. Parameter Hashing

`fp_params_hash` produces a stable 16-character hex string for cache key construction.

In [ ]:
h = fp_params_hash("ecfp", {"radius": 2})
print(f"Hash: {h}")
print(f"Length: {len(h)}")

# Also available as a property on the generator
print(f"Generator hash: {gen.params_hash}")
print(f"Match: {h == gen.params_hash}")

## 11. Caching for Repeated Lookups

Enable caching when you expect repeated SMILES/SELFIES queries.

In [ ]:
gen_cached = EmbeddedFingerprintGenerator(
    fp_type="ecfp", fp_size=2048, compression=16,
    fp_params={"radius": 2}, cache_size=1024
)

# First call computes and caches
_ = gen_cached.GetFingerprintFromSmiles("CCO")
# Second call hits cache
_ = gen_cached.GetFingerprintFromSmiles("CCO")

print(gen_cached.cache_info())
gen_cached.clear_cache()
print(f"After clear: {gen_cached.cache_info()}")

## 12. Compression Methods

fpembed supports six compression methods, selectable via the `method` parameter. The default is `"geometric". Methods fall into two categories:

- **Block-wise** (`geometric`, `linear`, `log`, `uniform`) — partition the fingerprint into blocks and apply weighted dot products. Fast, O(L) complexity. Optionally support bit-interleaving via `method_params={"interleave": True}`.
- **Global projection** (`hadamard`, `random_projection`) — project the entire fingerprint through a matrix transform so every output dimension depends on every input bit. Better information retention at high compression ratios.

In [ ]:
# Geometric (default)
gen_geo = EmbeddedFingerprintGenerator(
    fp_type="ecfp", fp_size=2048, compression=16, fp_params={"radius": 2}
)
emb_geo = gen_geo.GetFingerprintFromSmiles("CCO")
print(f"Geometric shape: {emb_geo.shape}")  # (128,)
print(f"Geometric first 5: {emb_geo[:5]}")

In [ ]:
# Linear with bit-interleaving
gen_lin = EmbeddedFingerprintGenerator(
    fp_type="ecfp", fp_size=2048, compression=16, fp_params={"radius": 2},
    method="linear", method_params={"interleave": True}
)
emb_lin = gen_lin.GetFingerprintFromSmiles("CCO")
print(f"Linear+interleave shape: {emb_lin.shape}")  # (128,)
print(f"Linear+interleave first 5: {emb_lin[:5]}")

In [ ]:
# Hadamard (SRHT) — global projection, requires power-of-2 fp_size
gen_had = EmbeddedFingerprintGenerator(
    fp_type="ecfp", fp_size=2048, compression=16, fp_params={"radius": 2},
    method="hadamard"
)
emb_had = gen_had.GetFingerprintFromSmiles("CCO")
print(f"Hadamard shape: {emb_had.shape}")  # (128,)
print(f"Hadamard first 5: {emb_had[:5]}")

In [ ]:
# Random projection (JL) — strongest distance-preservation guarantees
gen_rp = EmbeddedFingerprintGenerator(
    fp_type="ecfp", fp_size=2048, compression=16, fp_params={"radius": 2},
    method="random_projection"
)
emb_rp = gen_rp.GetFingerprintFromSmiles("CCO")
print(f"Random projection shape: {emb_rp.shape}")  # (128,)
print(f"Random projection first 5: {emb_rp[:5]}")

In [ ]:
# Comparison: different methods produce different embeddings but same shape
methods = {
    "geometric": emb_geo,
    "linear+interleave": emb_lin,
    "hadamard": emb_had,
    "random_projection": emb_rp,
}

for name, emb in methods.items():
    print(f"{name:25s} shape={emb.shape}  mean={emb.mean():.4f}  std={emb.std():.4f}")

# All shapes are identical
shapes = [emb.shape for emb in methods.values()]
assert all(s == shapes[0] for s in shapes), "Shape mismatch!"
print(f"\nAll methods produce shape {shapes[0]} ✓")

# But values differ
assert not np.array_equal(emb_geo, emb_had), "Expected different values"
print("Different methods produce different embeddings ✓")